In [ ]:
## Nikolay Vorontsov,
## Mushroom task
## Convert data to fit the format.

In [1]:
# Install necessary dependencies
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 3.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [2]:

from google.colab import drive



In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:

dataset_name = "/content/drive/MyDrive/Colab_Notebooks/Lingdig_mushroom/Nick/gemini_qa-pairs_with_words_7.1.2025.jsonl" #"UNDEFINED"



In [5]:
# I NEED THIS FORMAT
# 1. Load your dataset
data = [
    {"text": "The Eiffel Tower is located in Berlin, Germany.", "labels": [0, 0, 0, 0, 0, 0, 1, 1]},  # Hallucinated words: "Berlin", "Germany"
    {"text": "The capital of France is Paris.", "labels": [0, 0, 0, 0, 0, 0]},  # Correct sentence
    {"text": "The Amazon River flows through Asia.", "labels": [0, 0, 0, 0, 0, 1]},  # Hallucinated word: "Asia"
]


In [6]:
import json

# Function to convert JSONL data to desired format with exact match and split hallucinated words into single words
def convert_jsonl_to_word_dataset(jsonl_file_path, output_file_path):
    with open(jsonl_file_path, 'r', encoding='utf-8') as file, open(output_file_path, 'w', encoding='utf-8') as output_file:
        for line in file:
            record = json.loads(line.strip())

            # Extract information from the JSON object
            text = record["model_output_text"]
            hallucinated_words = record["hallucinated_words"]

            # Initialize labels for each word in the text
            words = text.split()
            labels = [0] * len(words)

            # Loop through hallucinated phrases for exact matches in the text
            for phrase in hallucinated_words:
                phrase_words = phrase.split()  # Split the hallucinated phrase into words
                phrase_length = len(phrase_words)

                # Search for exact matches of the hallucinated phrase (single word or multiple words)
                for i in range(len(words) - phrase_length + 1):
                    if words[i:i + phrase_length] == phrase_words:
                        # Label all words in the matched phrase
                        for j in range(i, i + phrase_length):
                            labels[j] = 1

            # Create the formatted record
            formatted_record = {"text": text, "labels": labels}

            # Write each record as a JSONL line
            output_file.write(json.dumps(formatted_record, ensure_ascii=False) + '\n')

# Example usage
input_jsonl_path = dataset_name  # Replace with your input file path
output_jsonl_path = 'data.jsonl'  # Replace with your output file path


In [8]:

convert_jsonl_to_word_dataset(input_jsonl_path, output_jsonl_path)
